In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

from IPython.display import display


In [2]:
# 辅助函数


def E(x, p):
    """Expected value of a discrete random variable."""
    return np.sum(x * p)


In [3]:
def f_S(N, g, f, m, k, b):
    """
    S_k = {
        (m - k) * g - f                  if k >= m - N
        (N * g - f) - (m - k - N) * b    if k <  m - N
    }
    """
    return np.where(k >= m - N, (m - k) * g - f, (N * g - f) - (m - k - N) * b)


def f_P(m, q, distribution):
    """B(m, q) or Poisson(m * q)"""
    k = np.arange(m + 1)
    if distribution == "poisson":
        rv = stats.poisson(m * q)
    elif distribution == "binom":
        rv = stats.binom(m, q)
    else:
        raise ValueError("Invalid distribution")
    return rv.pmf(k)  # type: ignore


In [4]:
def f_0(N, g, f, m_s, b, q, distribution):
    df = pd.DataFrame(columns=["E(S)/f", "P(n>=1)", "P(n>=5)"])
    df.index.name = "m"

    for m in m_s:
        k = np.arange(m + 1)
        n = np.clip(m - k - N, 0, None)
        p = f_P(m, q, distribution=distribution)
        S = f_S(N, g, f, m, k, b)
        E_S = E(S, p)
        E_S_div_f = E_S / f
        P_n_ge_1 = np.sum(p[n >= 1])
        P_n_ge_5 = np.sum(p[n >= 5])

        df.loc[m] = [E_S_div_f, P_n_ge_1, P_n_ge_5]

    return df


N = 300
g = 1
f = 0.6 * N * g
b = 0.2


for q, m_s in [
    (0.05, range(300, 327)),
    (0.1, range(300, 346)),
]:
    df = f_0(N, g, f, m_s, b, q, distribution="binom")
    print(f"q = {q:.2f}")
    display(df)


q = 0.05


,E(S)/f,P(n>=1),P(n>=5)
m,,,
300,0.583333,0.000000e+00,0.000000e+00
301,0.588611,1.971538e-07,0.000000e+00
302,0.593889,3.164319e-06,0.000000e+00
303,0.599166,2.556641e-05,0.000000e+00
304,0.604443,1.386970e-04,0.000000e+00
305,0.609717,5.685932e-04,1.605830e-07
306,0.614983,1.879777e-03,2.609474e-06
307,0.620226,5.223294e-03,2.134349e-05
308,0.625422,1.255515e-02,1.171992e-04


q = 0.10


,E(S)/f,P(n>=1),P(n>=5)
m,,,
300,0.500000,0.000000e+00,0.000000e+00
301,0.505000,1.686535e-14,0.000000e+00
302,0.510000,5.245124e-13,0.000000e+00
303,0.515000,8.189982e-12,0.000000e+00
304,0.520000,8.561123e-11,0.000000e+00
305,0.525000,6.740127e-10,1.106536e-14
306,0.530000,4.263262e-09,3.485587e-13
307,0.535000,2.256843e-08,5.512207e-12
308,0.540000,1.028497e-07,5.835354e-11


In [5]:
def f_1(N, g, f, b_s, q, distribution):
    df = pd.DataFrame(columns=["使E(S)/f最大的预订水平", "E(S)/f", "P(n>=5)"])
    df.index.name = "b/g"

    for b in b_s:
        m_s = range(N, N + 100)
        result = f_0(N, g, f, m_s, b, q, distribution=distribution)
        m_best = result["E(S)/f"].idxmax()
        E_S_div_f = result.loc[m_best, "E(S)/f"]
        P_n_ge_5 = result.loc[m_best, "P(n>=5)"]
        df.loc[b/g] = [m_best, E_S_div_f, P_n_ge_5]

    df = df.convert_dtypes()
    return df


N = 300
g = 1  # 假设票价为 1
f = 0.6 * N * g

b_s = [0.1, 0.2, 0.3, 0.4, 0.5]
for q in [0.05, 0.10, 0.15]:
    df = f_1(N, g, f, b_s, q, distribution="binom")
    print(f"q = {q:.2f}")
    display(df)


q = 0.05


,使E(S)/f最大的预订水平,E(S)/f,P(n>=5)
b/g,,,
0.1,321,0.662575,0.560894
0.2,320,0.65996,0.464191
0.3,319,0.657905,0.367487
0.4,318,0.656229,0.276542
0.5,317,0.654725,0.196466


q = 0.10


,使E(S)/f最大的预订水平,E(S)/f,P(n>=5)
b/g,,,
0.1,342,0.660953,0.729165
0.2,339,0.657273,0.552698
0.3,338,0.65436,0.487973
0.4,337,0.651931,0.422865
0.5,336,0.649872,0.359109


q = 0.15


,使E(S)/f最大的预订水平,E(S)/f,P(n>=5)
b/g,,,
0.1,364,0.659712,0.766555
0.2,361,0.65519,0.641107
0.3,359,0.651633,0.545045
0.4,357,0.648673,0.44487
0.5,356,0.646128,0.395158


In [6]:
N = 300
g = 1
f = 0.6 * N * g
b = 0.2


q = 0.05
m_s = range(300, 327)

df = f_0(N, g, f, m_s, b, q, distribution="poisson")
print(f"q = {q:.2f}")
display(df)

q = 0.1
m_s = range(300, 346)

df = f_0(N, g, f, m_s, b, q, distribution="poisson")
print(f"q = {q:.2f}")
display(df)


q = 0.05


,E(S)/f,P(n>=1),P(n>=5)
m,,,
300,0.583333,0.000000e+00,0.000000e+00
301,0.588611,2.909833e-07,0.000000e+00
302,0.593889,4.456349e-06,0.000000e+00
303,0.599166,3.446796e-05,0.000000e+00
304,0.604443,1.795792e-04,0.000000e+00
305,0.609716,7.092749e-04,2.382370e-07
306,0.614979,2.266246e-03,3.693874e-06
307,0.620217,6.104892e-03,2.892057e-05
308,0.625401,1.426958e-02,1.524956e-04


q = 0.10


,E(S)/f,P(n>=1),P(n>=5)
m,,,
300,0.500000,0.000000e+00,0.000000e+00
301,0.505000,8.467127e-14,0.000000e+00
302,0.510000,2.390349e-12,0.000000e+00
303,0.515000,3.399217e-11,0.000000e+00
304,0.520000,3.246630e-10,0.000000e+00
305,0.525000,2.343052e-09,5.675685e-14
306,0.530000,1.362897e-08,1.622841e-12
307,0.535000,6.656027e-08,2.337114e-11
308,0.540000,2.807272e-07,2.260339e-10


In [7]:
def f_S_多票价(N, g, f, m, k, b, j, r):
    """
    S_k = {
        j * r * g + (m - j - k) * g - f                  if k >= m - N
        j * r * g + (N - j) * g - f - (m - k - N) * b    if k <  m - N
    }
    """
    return np.where(k >= m - N,
                    j * r * g + (m - j - k) * g - f,
                    j * r * g + (N - j) * g - f - (m - k - N) * b)


In [8]:
def f_2(N, g, f, m_s, b, q, j, r, distribution):
    df = pd.DataFrame(columns=["E(S)/f", "P(n>=1)", "P(n>=5)"])
    df.index.name = "m"

    for m in m_s:
        k = np.arange(m - j + 1)
        n = np.clip(m - k - N, 0, None)
        p = f_P(m - j, q, distribution=distribution)
        S = f_S_多票价(N, g, f, m, k, b, j, r)
        E_S = E(S, p)
        E_S_div_f = E_S / f
        # print(S, p, E_S, E_S_div_f, sep="\n\n")
        P_n_ge_1 = np.sum(p[n >= 1])
        P_n_ge_5 = np.sum(p[n >= 5])

        df.loc[m] = [E_S_div_f, P_n_ge_1, P_n_ge_5]

    return df


N = 300
g = 1
r = 0.75
b = 0.2
q = 0.05
m_s = range(300, 327)

df_dict = {}
for j in [0, 50, 100, 150]:
    f = 0.6 * (j * r * g + (N - j) * g)
    df = f_2(N, g, f, m_s, b, q, j, r, distribution="binom")
    df_dict[j] = df
    print(f"j = {j}")
    display(df)


j = 0


,E(S)/f,P(n>=1),P(n>=5)
m,,,
300,0.583333,0.000000e+00,0.000000e+00
301,0.588611,1.971538e-07,0.000000e+00
302,0.593889,3.164319e-06,0.000000e+00
303,0.599166,2.556641e-05,0.000000e+00
304,0.604443,1.386970e-04,0.000000e+00
305,0.609717,5.685932e-04,1.605830e-07
306,0.614983,1.879777e-03,2.609474e-06
307,0.620226,5.223294e-03,2.134349e-05
308,0.625422,1.255515e-02,1.171992e-04


j = 50


,E(S)/f,P(n>=1),P(n>=5)
m,,,
300,0.594203,0.000000,0.000000
301,0.599710,0.000003,0.000000
302,0.605217,0.000035,0.000000
303,0.610723,0.000237,0.000000
304,0.616222,0.001092,0.000000
305,0.621704,0.003804,0.000002
306,0.627138,0.010720,0.000029
307,0.632471,0.025474,0.000199
308,0.637622,0.052559,0.000928


j = 100


,E(S)/f,P(n>=1),P(n>=5)
m,,,
300,0.606061,0.000000,0.000000
301,0.611818,0.000033,0.000000
302,0.617573,0.000368,0.000000
303,0.623316,0.002058,0.000000
304,0.629017,0.007776,0.000000
305,0.634615,0.022357,0.000027
306,0.640001,0.052248,0.000305
307,0.645024,0.103561,0.001737
308,0.649515,0.179431,0.006676


j = 150


,E(S)/f,P(n>=1),P(n>=5)
m,,,
300,0.619048,0.000000,0.000000
301,0.625076,0.000433,0.000000
302,0.631080,0.003700,0.000000
303,0.636990,0.016117,0.000000
304,0.642664,0.047778,0.000000
305,0.647886,0.108727,0.000352
306,0.652411,0.203198,0.003084
307,0.656036,0.326010,0.013739
308,0.658659,0.463735,0.041617


In [9]:
df = pd.DataFrame(columns=["使E(S)/f最大的预订水平", "E(S)/f", "P(n>=5)"])
df.index.name = "j"
for j, result in df_dict.items():
    m_best = result["E(S)/f"].idxmax()
    E_S_div_f = result.loc[m_best, "E(S)/f"]
    P_n_ge_5 = result.loc[m_best, "P(n>=5)"]
    df.loc[j] = [m_best, E_S_div_f, P_n_ge_5]

df = df.convert_dtypes()
df


,使E(S)/f最大的预订水平,E(S)/f,P(n>=5)
j,,,
0,320,0.65996,0.464191
50,317,0.66026,0.421485
100,314,0.660648,0.368932
150,311,0.661162,0.300953


In [10]:
N = 300
g = 1
r = 0.75
b = 0.2
q = 0.05
m_s = range(300, 327)

df_dict = {}
for j in [0, 50, 100, 150]:
    f = 0.6 * (j * r * g + (N - j) * g)
    df = f_2(N, g, f, m_s, b, q, j, r, distribution="poisson")
    df_dict[j] = df
    print(f"j = {j}")
    display(df)

df = pd.DataFrame(columns=["使E(S)/f最大的预订水平", "E(S)/f", "P(n>=5)"])
df.index.name = "j"
for j, result in df_dict.items():
    m_best = result["E(S)/f"].idxmax()
    E_S_div_f = result.loc[m_best, "E(S)/f"]
    P_n_ge_5 = result.loc[m_best, "P(n>=5)"]
    df.loc[j] = [m_best, E_S_div_f, P_n_ge_5]

df = df.convert_dtypes()
display(df)


j = 0


,E(S)/f,P(n>=1),P(n>=5)
m,,,
300,0.583333,0.000000e+00,0.000000e+00
301,0.588611,2.909833e-07,0.000000e+00
302,0.593889,4.456349e-06,0.000000e+00
303,0.599166,3.446796e-05,0.000000e+00
304,0.604443,1.795792e-04,0.000000e+00
305,0.609716,7.092749e-04,2.382370e-07
306,0.614979,2.266246e-03,3.693874e-06
307,0.620217,6.104892e-03,2.892057e-05
308,0.625401,1.426958e-02,1.524956e-04


j = 50


,E(S)/f,P(n>=1),P(n>=5)
m,,,
300,0.594203,0.000000,0.000000
301,0.599710,0.000004,0.000000
302,0.605217,0.000046,0.000000
303,0.610722,0.000300,0.000000
304,0.616220,0.001330,0.000000
305,0.621697,0.004474,0.000003
306,0.627121,0.012222,0.000038
307,0.632436,0.028254,0.000253
308,0.637557,0.056908,0.001136


j = 100


,E(S)/f,P(n>=1),P(n>=5)
m,,,
300,0.606061,0.000000,0.000000
301,0.611818,0.000043,0.000000
302,0.617572,0.000456,0.000000
303,0.623312,0.002449,0.000000
304,0.629006,0.008924,0.000000
305,0.634586,0.024863,0.000035
306,0.639941,0.056554,0.000380
307,0.644922,0.109571,0.002077
308,0.649364,0.186327,0.007698


j = 150


,E(S)/f,P(n>=1),P(n>=5)
m,,,
300,0.619048,0.000000,0.000000
301,0.625075,0.000526,0.000000
302,0.631075,0.004304,0.000000
303,0.636970,0.018047,0.000000
304,0.642614,0.051819,0.000000
305,0.647790,0.114868,0.000431
306,0.652263,0.210251,0.003606
307,0.655844,0.332036,0.015458
308,0.658443,0.467004,0.045334


,使E(S)/f最大的预订水平,E(S)/f,P(n>=5)
j,,,
0,320,0.659773,0.466745
50,317,0.660079,0.425193
100,314,0.660478,0.373934
150,311,0.661008,0.307306


In [11]:
def f_S_多票价2(N, g, f, m, k1, k2, b, j, r):
    """
    S_k = {
        j * r * g + (m - j - k2) * g - f                            if k1 + k2 >= m - N
        j * r * g + (N - j + k1) * g - f - (m - k1 - k2 - N) * b    if k1 + k2 <  m - N
    }
    """
    return np.where(k1 + k2 >= m - N,
                    j * r * g + (m - j - k2) * g - f,
                    j * r * g + (N - j + k1) * g - f - (m - k1 - k2 - N) * b)


def f_3(N, g, f, m_s, b, q1, q2, j, r, distribution):
    df = pd.DataFrame(columns=["E(S)/f", "P(n>=1)", "P(n>=5)"])
    df.index.name = "m"

    for m in m_s:
        k1 = np.arange(j + 1)
        P1 = f_P(j, q1, distribution=distribution)
        k2 = np.arange(m - j + 1)[:, np.newaxis]
        P2 = f_P(m - j, q2, distribution=distribution)[:, np.newaxis]
        k = k1 + k2
        P = P1 * P2
        S = f_S_多票价2(N, g, f, m, k1, k2, b, j, r)
        E_S = E(S, P)
        E_S_div_f = E_S / f
        n = np.clip(m - k - N, 0, None)
        P_n_ge_1 = np.sum(P[n >= 1])
        P_n_ge_5 = np.sum(P[n >= 5])

        df.loc[m] = [E_S_div_f, P_n_ge_1, P_n_ge_5]

    return df


N = 300
g = 1
r = 0.75
b = 0.2
q1 = 0.02
q2 = 0.05
m_s = range(300, 327)

df_dict = {}
for j in [0, 50, 100, 150]:
    f = 0.6 * (j * r * g + (N - j) * g)
    df = f_3(N, g, f, m_s, b, q1, q2, j, r, distribution="binom")
    df_dict[j] = df
    print(f"j = {j}")
    display(df)


j = 0


,E(S)/f,P(n>=1),P(n>=5)
m,,,
300,0.583333,0.000000e+00,0.000000e+00
301,0.588611,1.971538e-07,0.000000e+00
302,0.593889,3.164319e-06,0.000000e+00
303,0.599166,2.556641e-05,0.000000e+00
304,0.604443,1.386970e-04,0.000000e+00
305,0.609717,5.685932e-04,1.605830e-07
306,0.614983,1.879777e-03,2.609474e-06
307,0.620226,5.223294e-03,2.134349e-05
308,0.625422,1.255515e-02,1.171992e-04


j = 50


,E(S)/f,P(n>=1),P(n>=5)
m,,,
300,0.594203,0.000000e+00,0.000000e+00
301,0.599710,9.331011e-07,0.000000e+00
302,0.605217,1.354806e-05,0.000000e+00
303,0.610724,9.915052e-05,0.000000e+00
304,0.616228,4.878933e-04,0.000000e+00
305,0.621723,1.817000e-03,7.600167e-07
306,0.627192,5.466229e-03,1.118698e-05
307,0.632605,1.384747e-02,8.298471e-05
308,0.637906,3.040935e-02,4.138216e-04


j = 100


,E(S)/f,P(n>=1),P(n>=5)
m,,,
300,0.606061,0.000000,0.000000
301,0.611818,0.000004,0.000000
302,0.617575,0.000057,0.000000
303,0.623330,0.000376,0.000000
304,0.629076,0.001662,0.000000
305,0.634793,0.005569,0.000004
306,0.640443,0.015108,0.000047
307,0.645955,0.034605,0.000316
308,0.651224,0.068914,0.001416


j = 150


,E(S)/f,P(n>=1),P(n>=5)
m,,,
300,0.619048,0.000000,0.000000
301,0.625079,0.000021,0.000000
302,0.631109,0.000239,0.000000
303,0.637130,0.001389,0.000000
304,0.643121,0.005441,0.000000
305,0.649032,0.016212,0.000017
306,0.654770,0.039245,0.000198
307,0.660204,0.080513,0.001171
308,0.665167,0.144235,0.004663


In [12]:
df = pd.DataFrame(columns=["使E(S)/f最大的预订水平", "E(S)/f", "P(n>=5)"])
df.index.name = "j"
for j, result in df_dict.items():
    m_best = result["E(S)/f"].idxmax()
    E_S_div_f = result.loc[m_best, "E(S)/f"]
    P_n_ge_5 = result.loc[m_best, "P(n>=5)"]
    df.loc[j] = [m_best, E_S_div_f, P_n_ge_5]

df = df.convert_dtypes()
df


,使E(S)/f最大的预订水平,E(S)/f,P(n>=5)
j,,,
0,320,0.65996,0.464191
50,318,0.66582,0.419186
100,316,0.672214,0.369389
150,314,0.679222,0.314267
